<a href="https://colab.research.google.com/github/DYLPICKLEROHAN/Chloe-s-website/blob/Website/2023_big_data_analysis_and_industry_project/Assessment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Connecting to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# installing pyspark module
!pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.4/281.4 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 KB 19.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.2-py2.py3-none-any.whl size=281824028 sha256=7eaa4760d5fa1d8d0fc3cf2ec436dada30d144f7846df3485566851fc199ead2
  Stored in directory: /root/.cache/pip/wheels/6c/e3/9b/0525ce8a69478916513509d43693511463c6468db0de237c86
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7


# Assessment 2
### Exercise 1 - PageRank

PageRank algorythms use the edges and nodes of a network to determine the probability of ending up on a given page after some amount of time/transitions - effectively reaching a steady state in a markov chain. The results are then listed in order of highest liklihood to lowest likelihood.

This can be troublesome if the web crawler gets stuck in a loop/ segment of the network (inflating the liklihood of specific pages) or if there are dangling nodes in the network that don't link back into the network. There are things that can be done about these as will be seen below. But first lets look at a 'first principles' implementation with a simple network.

In [ ]:
# Importing modules
import sys
import numpy as np
from pyspark import SparkContext, SparkConf
from operator import add
from pyspark.sql.functions import datediff

conf = SparkConf().setAppName("assessment2").set("spark.hadoop.fs.permissions.umask-mode", "0000")
sc = SparkContext(conf=conf)

In [ ]:
# PageRank algorythm
## The network outlined in the textbook is:
network =sc.parallelize([('A', ['B', 'C', 'D']),
                         ('B', ['A', 'D']),
                         ('C', ['A']),
                         ('D', ['B', 'C'])])
print("network from chapter:", network.collect())

## The initial page ranks will be...
node_count = network.count()
v_initial = network.map(lambda node: (node[0], 1/node_count))
print("v initial =", v_initial.collect())

# Prepping the transition matrix
transition_matrix_prep = network.flatMap(lambda node: [((node[0], neighbour), 1/len(node[1])) for neighbour in node[1] if len(node[1])>0])
print(transition_matrix_prep.collect())
# Reducing to cover the case in which there are two edges connecting the same two nodes
M = transition_matrix_prep.reduceByKey(add)

network from chapter: [('A', ['B', 'C', 'D']), ('B', ['A', 'D']), ('C', ['A']), ('D', ['B', 'C'])]
v initial = [('A', 0.25), ('B', 0.25), ('C', 0.25), ('D', 0.25)]
[(('A', 'B'), 0.3333333333333333), (('A', 'C'), 0.3333333333333333), (('A', 'D'), 0.3333333333333333), (('B', 'A'), 0.5), (('B', 'D'), 0.5), (('C', 'A'), 1.0), (('D', 'B'), 0.5), (('D', 'C'), 0.5)]


In [ ]:
v_last_chainlink = v_initial
matrix = M.map(lambda x: (x[0][0], (x[0][1], x[1])))
for i in range(10): # Arbitrary choice
    v_next_chainlink = matrix.join(v_last_chainlink).map(lambda x: (x[1][0][0], x[1][0][1] * x[1][1])).reduceByKey(add) # (Destination, pagerank score)
    print(v_next_chainlink.collect())
    v_last_chainlink = v_next_chainlink

[('B', 0.20833333333333331), ('A', 0.375), ('C', 0.20833333333333331), ('D', 0.20833333333333331)]
[('C', 0.22916666666666666), ('A', 0.3125), ('B', 0.22916666666666666), ('D', 0.22916666666666666)]
[('B', 0.21875), ('C', 0.21875), ('A', 0.34375), ('D', 0.21875)]
[('A', 0.328125), ('B', 0.22395833333333331), ('D', 0.22395833333333331), ('C', 0.22395833333333331)]
[('A', 0.3359375), ('D', 0.22135416666666666), ('C', 0.22135416666666666), ('B', 0.22135416666666666)]
[('D', 0.22265625), ('A', 0.33203125), ('C', 0.22265625), ('B', 0.22265625)]
[('B', 0.22200520833333331), ('D', 0.22200520833333331), ('C', 0.22200520833333331), ('A', 0.333984375)]
[('C', 0.22233072916666666), ('D', 0.22233072916666666), ('A', 0.3330078125), ('B', 0.22233072916666666)]
[('A', 0.33349609375), ('D', 0.22216796875), ('B', 0.22216796875), ('C', 0.22216796875)]
[('B', 0.22224934895833331), ('C', 0.22224934895833331), ('A', 0.333251953125), ('D', 0.22224934895833331)]


And there you have it, a simple page rank algorythm.
Now lets build on this in a way that manages dangling nodes or spider traps.
The simple case was simply v' = Mv.
The more complex case involves the equation $\mathbf{v}^{\prime}=\beta M \mathbf{v}+(1-\beta) \mathbf{e} / n$, it adds randomness to manage dangling nodes and spider traps.

This randomness also reduces the amount of iterations required to reach a steady state.

In [ ]:
# Complex pageRank algorythm
beta = 0.85 # Adds some randomness to handle spider traps
v_last_chainlink = v_initial
matrix = M.map(lambda x: (x[0][0], (x[0][1], x[1])))
for i in range(10): # Arbitrary choice
    v_next_chainlink_step = matrix.join(v_last_chainlink).map(lambda x: (x[1][0][0], beta * x[1][0][1] * x[1][1])).reduceByKey(add) # Destination and liklihood
    v_next_chainlink = v_next_chainlink_step.map(lambda x:(x[0], x[1]+ (1-beta)*(1/node_count)))
    print(v_next_chainlink.collect())
    v_last_chainlink = v_next_chainlink

[('B', 0.21458333333333332), ('A', 0.35624999999999996), ('C', 0.21458333333333332), ('D', 0.21458333333333332)]
[('C', 0.22963541666666665), ('A', 0.31109374999999995), ('B', 0.22963541666666665), ('D', 0.22963541666666665)]
[('B', 0.22323828124999998), ('C', 0.22323828124999998), ('A', 0.33028515625), ('D', 0.22323828124999998)]
[('A', 0.3221288085937499), ('B', 0.22595706380208333), ('D', 0.22595706380208333), ('C', 0.22595706380208333)]
[('A', 0.3255952563476562), ('D', 0.2248015812174479), ('C', 0.2248015812174479), ('B', 0.2248015812174479)]
[('D', 0.22529266131591794), ('A', 0.3241220160522461), ('C', 0.22529266131591794), ('B', 0.22529266131591794)]
[('B', 0.2250839522740682), ('D', 0.2250839522740682), ('C', 0.2250839522740682), ('A', 0.3247481431777953)]
[('C', 0.22517265361685432), ('D', 0.22517265361685432), ('A', 0.32448203914943696), ('B', 0.22517265361685432)]
[('A', 0.32459513336148926), ('D', 0.22513495554617022), ('B', 0.22513495554617022), ('C', 0.22513495554617022)]

Now to wrap it into a function and apply it to the network from part 2 of section 1:

In [ ]:
from pyspark import SparkContext, SparkConf

def page_rank(network, beta=0.85, iterations=10):
    # Initialize page ranks
    node_count = network.count()
    v_initial = network.map(lambda node: (node[0], 1/node_count))
    print("v initial =", v_initial.collect(), "\n")

    # Create transition matrix
    transition_matrix_prep = network.flatMap(lambda node: [((node[0], neighbour), 1/len(node[1])) for neighbour in node[1] if len(node[1])>0])
    M = transition_matrix_prep.reduceByKey(add)


    # Run PageRank algorithm
    v_last_chainlink = v_initial
    matrix = M.map(lambda x: (x[0][0], (x[0][1], x[1])))
    for i in range(iterations):
        v_next_chainlink_step = matrix.join(v_last_chainlink).map(lambda x: (x[1][0][0], beta * x[1][0][1] * x[1][1])).reduceByKey(add)
        v_next_chainlink = v_next_chainlink_step.map(lambda x:(x[0], x[1]+ (1-beta)*(1/node_count)))
        v_last_chainlink = v_next_chainlink
        print(v_next_chainlink.collect())

page_rank(network)

v initial = [('A', 0.25), ('B', 0.25), ('C', 0.25), ('D', 0.25)] 

[('B', 0.21458333333333332), ('A', 0.35624999999999996), ('C', 0.21458333333333332), ('D', 0.21458333333333332)]
[('C', 0.22963541666666665), ('A', 0.31109374999999995), ('B', 0.22963541666666665), ('D', 0.22963541666666665)]
[('B', 0.22323828124999998), ('C', 0.22323828124999998), ('A', 0.33028515625), ('D', 0.22323828124999998)]
[('A', 0.3221288085937499), ('B', 0.22595706380208333), ('D', 0.22595706380208333), ('C', 0.22595706380208333)]
[('A', 0.3255952563476562), ('D', 0.2248015812174479), ('C', 0.2248015812174479), ('B', 0.2248015812174479)]
[('D', 0.22529266131591794), ('A', 0.3241220160522461), ('C', 0.22529266131591794), ('B', 0.22529266131591794)]
[('B', 0.2250839522740682), ('D', 0.2250839522740682), ('C', 0.2250839522740682), ('A', 0.3247481431777953)]
[('C', 0.22517265361685432), ('D', 0.22517265361685432), ('A', 0.32448203914943696), ('B', 0.22517265361685432)]
[('A', 0.32459513336148926), ('D', 0.22513495

In [ ]:
# Applying to other network
## importing file
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Pageranking').getOrCreate()
df = spark.read.csv('/content/drive/MyDrive/aabigdatafiles/assessment2/web-Google.txt', sep = '\t', comment = '#')
df.show()

+------+------+
|   _c0|   _c1|
+------+------+
|     0| 11342|
|     0|824020|
|     0|867923|
|     0|891835|
| 11342|     0|
| 11342| 27469|
| 11342| 38716|
| 11342|309564|
| 11342|322178|
| 11342|387543|
| 11342|427436|
| 11342|538214|
| 11342|638706|
| 11342|645018|
| 11342|835220|
| 11342|856657|
| 11342|867923|
| 11342|891835|
|824020|     0|
|824020| 91807|
+------+------+
only showing top 20 rows



In [ ]:
## Formatting
network = df.rdd.map(lambda x: (x[0], x[1])).groupByKey().mapValues(tuple)
print(network.first())

('0', ('11342', '824020', '867923', '891835'))


In [ ]:
# Applying Pagerank algorythm
def page_rank(network, beta=0.85, iterations=10):
    # Initialize page ranks
    node_count = network.count()
    v_initial = network.map(lambda node: (node[0], 1/node_count))

    # Create transition matrix
    transition_matrix_prep = network.flatMap(lambda node: [((node[0], neighbour), 1/len(node[1])) for neighbour in node[1] if len(node[1])>0])
    M = transition_matrix_prep.reduceByKey(add)


    # Run PageRank algorithm
    v_last_chainlink = v_initial
    matrix = M.map(lambda x: (x[0][0], (x[0][1], x[1])))
    for i in range(iterations):
        v_next_chainlink_step = matrix.join(v_last_chainlink).map(lambda x: (x[1][0][0], beta * x[1][0][1] * x[1][1])).reduceByKey(add)
        v_next_chainlink = v_next_chainlink_step.map(lambda x:(x[0], x[1]+ (1-beta)*(1/node_count)))
        v_last_chainlink = v_next_chainlink
        return v_next_chainlink

results = page_rank(network)


In [ ]:
results_sorted = results.sortBy(lambda x: x[1], False)

In [ ]:
top_10 = results_sorted.take(10)
top_10

[('163075', 0.0012193243696203252),
 ('537039', 0.0011715596620668683),
 ('597621', 0.0011617996313351524),
 ('605856', 0.0010969347509666855),
 ('885605', 0.0010506736402784453),
 ('751384', 0.0010158153575416345),
 ('908351', 0.0009763013419350915),
 ('504140', 0.0009521489100819803),
 ('32163', 0.0009472060091690455),
 ('173976', 0.000943672515896725)]

In [ ]:
import pandas as pd
output = pd.DataFrame(top_10)
output.to_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/page_rank_output.csv")

### Exercise 2 - Frequent Itemsets
There are two algorythms I am to provide, the simple and the more complex SON method. The simple algorythm simply takes a random sample of the dataset and assumes it is representative enough to determine the most significant associations in the real world. The SON algorythm uses the entire dataset, dividing it into chunks and then comparing all chunks to determine frequent itemsets.

The simple algorythm can be seen below:

#### Part 1:

In [ ]:
import itertools
import pandas as pd
#Frequent itemsets - The simple algorythm
test_set = [('A', ['Apple', 'Orange', 'Pear', 'Strawberry']),
            ('B', ['Apple', 'Orange']),
            ('C', ['Orange', 'Pear', 'Strawberry']),
            ('D', ['Apple', 'Orange', 'Pear', 'Strawberry']),
            ('E', ['Apple', 'Orange', 'Pear'])]

df = pd.DataFrame(test_set)

# Assume the test set is some random sample of the entire dataset
# Don't parallelize
baskets = []
for basket in df[1]:
  baskets.append(basket)



def alternative_method(baskets, support_threshold, max_size):
    items = {}
    for i, basket in enumerate(baskets):
      for item in basket:
          if item not in items:
              items[item] = {i}
          else:
              items[item].add(i)

    frequent_itemsets = {}
    basket_counts = [set() for _ in baskets]
    for item, basket_IDs in items.items():
        if len(basket_IDs) >= support_threshold:
            frequent_itemsets[(item)] = len(basket_IDs)  # Store itemsets of size 1
            for basket_ID in basket_IDs:
                basket_counts[basket_ID].add(item)

    for size in range(2, max_size+1):  # Range up to size len(items) + 1 to include size 1
        frequent_itemsets_size = {}
        for itemset in itertools.combinations(items.keys(), size):
            common_basket_IDs = set(items[itemset[0]])
            for item in itemset:
                common_basket_IDs = common_basket_IDs.intersection(set(items[item]))
            # Add the itemset to the frequent itemsets if its support is at least min_support
            support = 0
            for basket_ID in common_basket_IDs:
                if set(itemset).issubset(basket_counts[basket_ID]):
                    support += 1
            if support >= support_threshold:
                frequent_itemsets_size[itemset] = support
        frequent_itemsets.update(frequent_itemsets_size)
        if not frequent_itemsets_size:  # If no frequent itemsets of a particular size, stop iterating
            print("No frequent itemset of size", size)
            break
    return frequent_itemsets

alternative_method(baskets, 2, 4)


{'Apple': 4,
 'Orange': 5,
 'Pear': 4,
 'Strawberry': 3,
 ('Apple', 'Orange'): 4,
 ('Apple', 'Pear'): 3,
 ('Apple', 'Strawberry'): 2,
 ('Orange', 'Pear'): 4,
 ('Orange', 'Strawberry'): 3,
 ('Pear', 'Strawberry'): 3,
 ('Apple', 'Orange', 'Pear'): 3,
 ('Apple', 'Orange', 'Strawberry'): 2,
 ('Apple', 'Pear', 'Strawberry'): 2,
 ('Orange', 'Pear', 'Strawberry'): 3,
 ('Apple', 'Orange', 'Pear', 'Strawberry'): 2}

#### Part 2

In [ ]:
# SON method attempt 1
# The difference is parallelization of the whole dataset, not just a random sample
rdd = spark.sparkContext.parallelize(df[1].to_numpy())
print(rdd.first(), "\n")

# Formatting the RDD
baskets = rdd.map(tuple)
print(baskets.first())

# Initializing with itemsets of size = 1
all_items = baskets.flatMap(lambda basket: [(item, 1) for item in basket])
all_items = all_items.reduceByKey(lambda a, b: a + b)
print(all_items.first())

# Gathering items with support of 3 or higher
support_threshold = 2
frequent_items = all_items.filter(lambda all_items: all_items[1] >= support_threshold)
print(frequent_items.collect())

# Making a reference of the frequent itemsets available to all nodes (pickling error resolved by converting to lists)
frequent_items_broadcast = spark.sparkContext.broadcast(list(frequent_items.collectAsMap().keys()))

# filtering each basket to relevant items
frequent_baskets = baskets.filter(lambda basket: any(item in frequent_items_broadcast.value for item in basket))

# Of these baskets, we need to find all the combinations
size = 2
combinations_in_basket = frequent_baskets.flatMap(lambda basket: itertools.combinations(basket, size))

# Map reducing
combinations_in_basket = combinations_in_basket.map(lambda combination: (combination, size-1))
print("combinations_in_basket", combinations_in_basket.take(1))
combination_count = combinations_in_basket.reduceByKey(lambda a, b: a + b)
print("combination_count", combination_count.first())

# Checking relevant pairs
combinations_above_suport = combination_count.filter(lambda combinations: combinations[1] >= support_threshold)
print(combinations_above_suport.collect())

['Apple', 'Orange', 'Pear', 'Strawberry'] 

('Apple', 'Orange', 'Pear', 'Strawberry')
('Apple', 4)
[('Apple', 4), ('Orange', 5), ('Strawberry', 3), ('Pear', 4)]
combinations_in_basket [(('Apple', 'Orange'), 1)]
combination_count (('Apple', 'Orange'), 4)
[(('Apple', 'Orange'), 4), (('Apple', 'Strawberry'), 2), (('Orange', 'Strawberry'), 3), (('Apple', 'Pear'), 3), (('Orange', 'Pear'), 4), (('Pear', 'Strawberry'), 3)]


In [ ]:
# Building on the above:
# The difference is parallelization of the whole dataset, not just a random sample
rdd = spark.sparkContext.parallelize(df[1].to_numpy())

# Formatting the RDD
baskets = rdd.map(tuple)

def freq_itemsets_son(baskets, support):
    # Initializing with itemsets of size = 1
    all_items = baskets.flatMap(lambda basket: [(item, 1) for item in basket])
    all_items = all_items.reduceByKey(lambda a, b: a + b)

    # Gathering items with support of x or higher
    frequent_items = all_items.filter(lambda all_items: all_items[1] >= support_threshold)

    # Making a reference of the frequent itemsets available to all nodes (pickling error resolved by converting to lists)
    frequent_items_broadcast = spark.sparkContext.broadcast(list(frequent_items.collectAsMap().keys()))

    # Keeping baskets containing items above support threshold
    frequent_baskets = baskets.filter(lambda basket: any(item in frequent_items_broadcast.value for item in basket))

    # Of these baskets, we need to find all the combinations
    fis_dict = {}
    size = 1
    condition = frequent_baskets.count()
    while condition > 1:
        size += 1
        # Gathering combinations of items in baskets (initially for itemsets with two items)
        combinations_in_basket = frequent_baskets.flatMap(lambda basket: itertools.combinations(basket, size))
        # Map reducing
        combinations_in_basket = combinations_in_basket.map(lambda combination: (combination, 1))
        combination_count = combinations_in_basket.reduceByKey(lambda a, b: a + b)

        # Checking relevant pairs
        combinations_above_support = combination_count.filter(lambda combinations: combinations[1] >= support_threshold)
        condition = combinations_above_support.count()
        for combination in combinations_above_support.collect():
          fis_dict[combination[0]]= combination[1]
    return fis_dict

In [ ]:
freq_itemsets_son(baskets, 2)

{('Apple', 'Orange'): 4,
 ('Apple', 'Strawberry'): 2,
 ('Orange', 'Strawberry'): 3,
 ('Apple', 'Pear'): 3,
 ('Orange', 'Pear'): 4,
 ('Pear', 'Strawberry'): 3,
 ('Apple', 'Orange', 'Pear'): 3,
 ('Apple', 'Pear', 'Strawberry'): 2,
 ('Orange', 'Pear', 'Strawberry'): 3,
 ('Apple', 'Orange', 'Strawberry'): 2,
 ('Apple', 'Orange', 'Pear', 'Strawberry'): 2}

#### Part 3

Note: I am new to spark and big data, I've come to the end of this assignment and realized I could have just used the alternative apriori method in the Son method instead of just rewriting the code in pyspark (which was difficult and took me days to figure out). That way the son method would have had the added efficiency and memory reduction of the former method. I wish I had the time to fix this, but I've given every last second I had to this assignment and I'm out of time.

Comparing the two methods I have here, the apriori algorythm is faster when using a support of two and only 0.01% of the data. But as the data increases it becomes obvious that the SON method is far superior.

The SON Method is by far more thorough and much faster at processing, given the amount of tasks it is processing at any given time as a result of parallelization.

In [ ]:
# Importing T10I4D100K to correct format
import pandas as pd
T10I4D100K = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/T10I4D100K.csv", header = None, index_col= False)

T10I4D100K_baskets = []
for basket in T10I4D100K[0]:
  basket = basket.split()
  T10I4D100K_baskets.append(basket)

In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
percent_1 = int(round(len(T10I4D100K_baskets)*0.0001-1,0))                           #-1 because it starts at 0
sample_chunk = T10I4D100K_baskets[:percent_1]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 2, 3)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(T10I4D100K_baskets)
T10I4D100K_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(T10I4D100K_baskets, 100)

In [ ]:
# Importing T40I10D100K to correct format
import pandas as pd
T40I10D100K = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/T40I10D100K.csv", header = None, index_col= False)

T40I10D100K_baskets = []
for basket in T40I10D100K[0]:
  basket = basket.split()
  T40I10D100K_baskets.append(basket)


In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
chunk_size = int(round(len(T40I10D100K_baskets)*0.0001-1,0))                           #-1 because it starts at 0
sample_chunk = T40I10D100K_baskets[:chunk_size]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 10, 10)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(T40I10D100K_baskets)
T40I10D100K_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(T40I10D100K_baskets, 10)

In [ ]:
# Importing chess to correct format
import pandas as pd
chess = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/chess.csv", header = None, index_col= False)

chess_baskets = []
for basket in chess[0]:
  basket = basket.split()
  chess_baskets.append(basket)

In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
percent_1 = int(round(len(chess_baskets)*0.001-1,0))                           #-1 because it starts at 0
sample_chunk = chess_baskets[:percent_1]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 10, 10)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(chess_baskets)
chess_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(chess_baskets, 10)

In [ ]:
# Importing mushroom to correct format
import pandas as pd
mushroom = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/mushroom.csv", header = None, index_col= False)

mushroom_baskets = []
for basket in mushroom[0]:
  basket = basket.split()
  mushroom_baskets.append(basket)

In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
percent_1 = int(round(len(mushroom_baskets)*0.0001-1,0))                           #-1 because it starts at 0
sample_chunk = mushroom_baskets[:percent_1]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 2, 10)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(mushroom_baskets)
mushroom_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(mushroom_baskets, 10)

In [ ]:
# Importing pumsb to correct format
import pandas as pd
pumsb = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/pumsb.csv", header = None, index_col= False)

pumsb_baskets = []
for basket in pumsb[0]:
  basket = basket.split()
  pumsb_baskets.append(basket)

In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
percent_1 = int(round(len(pumsb_baskets)*0.0001-1,0))                           #-1 because it starts at 0
sample_chunk = pumsb_baskets[:percent_1]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 2, 10)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(pumsb_baskets)
pumsb_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(pumsb_baskets, 10)

In [ ]:
# Importing pumsb_star to correct format
import pandas as pd
pumsb_star = pd.read_csv("/content/drive/MyDrive/aabigdatafiles/assessment2/pumsb_star.csv", header = None, index_col= False)

pumsb_star_baskets = []
for basket in pumsb_star[0]:
  basket = basket.split()
  pumsb_star_baskets.append(basket)

In [ ]:
# Taking a sample chunk of the first 5% baskets to demonstrate
percent_1 = int(round(len(pumsb_star_baskets)*0.0001-1,0))                           #-1 because it starts at 0
sample_chunk = pumsb_star_baskets[:percent_1]

# Running the apriori method on a chunk
alternative_method(sample_chunk, 2, 10)

In [ ]:
# Formatting the RDD
rdd = spark.sparkContext.parallelize(pumsb_star_baskets)
pumsb_star_baskets = rdd.map(tuple)

# Running the SON method on all
freq_itemsets_son(pumsb_star_baskets, 10)

#### Step 4

I don't have the time to run this as it took me quite some time to code this (and I bought a house this week!!), but essentially what you will find is that as you increase the percentage of the data used, the processing time will increase. In exchange for this increased time, the results will approach that achieved by the SON method.

with the apriori method, there is a trade off between time and accuracy. The SON method breaks this tradeoff down by seperating the tasks so they can run concurrently.

### Exercise 3 - Clustering
#### Part 1: By closest centroid


In [ ]:
points = [1, 4, 11, 16, 25, 36, 49, 64, 81]

def distance(points):
    # 1 dimensional, so we only need consider the distance between neighbours
    # Lets first create a dictionary of cluster, location pairs
    cluster_dict = {point: [point] for point in points}

    while len(cluster_dict) > 1:
      # In the case of average centroids, we look at the midpoint
      pairs = {}
      for i, k in enumerate(cluster_dict):
            if i < len(cluster_dict)-1:
                midpoint = (cluster_dict[k][0] + cluster_dict[list(cluster_dict.keys())[i+1]][0]) / 2
                cluster_position = midpoint + k
                pairs[(k, list(cluster_dict.keys())[i+1])] = cluster_position

      # Finding the minimum distance between any two clusters
      min_distance = min(pairs, key=pairs.get)

      # Clustering step and assigning cluster position
      midpoint = (cluster_dict[min_distance[0]][0] + cluster_dict[min_distance[1]][0]) / 2
      merged = cluster_dict[min_distance[0]] + cluster_dict[min_distance[1]]
      del cluster_dict[min_distance[0]]
      del cluster_dict[min_distance[1]]
      cluster_dict[midpoint] = merged
      print(list(cluster_dict.values()), "\n")

distance(points)

[[11], [16], [25], [36], [49], [64], [81], [1, 4]] 

[[25], [36], [49], [64], [81], [1, 4], [11, 16]] 

[[25], [36], [49], [64], [81], [1, 4, 11, 16]] 

[[49], [64], [81], [1, 4, 11, 16], [25, 36]] 

[[49], [64], [81], [1, 4, 11, 16, 25, 36]] 

[[81], [1, 4, 11, 16, 25, 36], [49, 64]] 

[[81], [1, 4, 11, 16, 25, 36, 49, 64]] 

[[81, 1, 4, 11, 16, 25, 36, 49, 64]] 



#### Part 2: By closest point in or outside a cluster (more nearest neighbours like)

In [ ]:
points = [1, 4, 11, 16, 25, 36, 49, 64, 81]

# Initialize each point in its own cluster
cluster_dict = {p : [p] for p in points}

while len(cluster_dict) > 1:
# Calculate the distance between the closest points in or out of clusters
    pairs = {}
    for i, k in enumerate(cluster_dict):
        if i < len(cluster_dict)-1:
          # Resetting to a larger number
            minimum_dist = float('inf')
            # Searching through for the minimum distance
            for point1 in cluster_dict[k]:
                for point2 in cluster_dict[list(cluster_dict.keys())[i+1]]:
                    dist = abs(point2 - point1)
                    if dist < minimum_dist:
                        minimum_dist = dist
            # Merging the cluster and setting the value to the minimum_distance
            pairs[(k, list(cluster_dict.keys())[i+1])] = minimum_dist
    #print("Potential pairs", pairs)
    # Find the pair with the smallest distance
    pair_min = min(pairs, key=pairs.get)

    # Adding a merged cluster and removing their constituent pairs
    new_cluster = cluster_dict[pair_min[0]] + cluster_dict[pair_min[1]]
    #print("New cluster", new_cluster)
    del cluster_dict[pair_min[0]]
    del cluster_dict[pair_min[1]]
    cluster_dict[tuple(new_cluster)] = new_cluster
    # Note: The values are the points assoicated with a given cluster/key
    print("Updated clusters", list(cluster_dict.keys()), "\n")


Updated clusters [11, 16, 25, 36, 49, 64, 81, (1, 4)] 

Updated clusters [25, 36, 49, 64, 81, (1, 4), (11, 16)] 

Updated clusters [25, 36, 49, 64, 81, (1, 4, 11, 16)] 

Updated clusters [49, 64, 81, (1, 4, 11, 16), (25, 36)] 

Updated clusters [49, 64, 81, (1, 4, 11, 16, 25, 36)] 

Updated clusters [81, (1, 4, 11, 16, 25, 36), (49, 64)] 

Updated clusters [81, (1, 4, 11, 16, 25, 36, 49, 64)] 

Updated clusters [(81, 1, 4, 11, 16, 25, 36, 49, 64)] 

